# Effective Scintillator Propagation Speed with Time-Walk Correction ($c_{\mathrm{scint}}^{\mathrm{TW}}$)
## Comprehensive Analysis & Direct Comparison: Standard ($c_{\mathrm{scint}}$) vs. Time-Walk Corrected ($c_{\mathrm{scint}}^{\mathrm{TW}}$)

This notebook investigates the impact of SiPM **Time-Walk corrections** on the effective scintillation light propagation speed ($c_{\mathrm{scint}}$) across all available SND@LHC data-taking periods:
- **Category II (2022)**: 33 physics runs (Runs 5119 to 5408)
- **Category III (2023)**: 10 physics runs (Runs 6079 to 6667)
- **Category III (2025)**: 10 physics runs (Runs 11124 to 12217)
- *(Cat III 2024 & 2026 auto-ingested as soon as their partition extractions complete)*

---

### Key Investigations:
1. **$c_{\mathrm{scint}}^{\mathrm{TW}}$ Stability vs. Run Number:** Run-by-run evolution for Veto, Upstream (US), and Downstream (DS) SiPM channels.
2. **Readout Symmetry & Left–Right Consistency:** Impact of time-walk corrections on the $\Delta(L - R)$ readout speed asymmetry.
3. **Plane-by-Plane Profiles:** Upstream planes 1–5 stability and systematic trends before vs. after time-walk compensation.
4. **Fit Quality & Residuals ($\chi^2/\mathrm{ndf}$):** Goodness-of-fit distributions across planes.
5. **Channel-by-Channel Direct Differential ($\Delta c = c_{\mathrm{scint}}^{\mathrm{TW}} - c_{\mathrm{scint}}^{\mathrm{orig}}$):** Direct channel-matched shifts, correlation plots, and fractional speed increases across all detector subsystems.

In [ ]:
# Fix JupyROOT post-execute hook crash (ROOT::Experimental::RCanvas::GetCanvases ValueError)
try:
    import ROOT
    try:
        import ROOT._jupyroot.helpers.utils as _jutils
        _jutils.GetRCanvasDrawers = lambda: []
    except (ImportError, AttributeError):
        pass
except ImportError:
    pass

import os
import glob
import json
import concurrent.futures
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats

try:
    import ddfUtils.mpl_styling as dms
except ImportError:
    pass

print("Environment initialized successfully.")

---
## 1. Dual Ingestion: Standard $c_{\mathrm{scint}}$ and Time-Walk Corrected $c_{\mathrm{scint}}^{\mathrm{TW}}$

We load fit JSON files in parallel using `ThreadPoolExecutor` from:
- **Standard (Original):** `/eos/user/i/idioniso/Timing/cscint/fits/run*_cscint.json`
- **Time-Walk Corrected:** `/eos/user/i/idioniso/Timing/cscint_tw/fits/run*_cscint.json`

In [ ]:
fits_dir_orig = "/eos/user/i/idioniso/Timing/cscint/fits"
fits_dir_tw = "/eos/user/i/idioniso/Timing/cscint_tw/fits"

def get_run_period(run: int) -> str:
    """Classifies run into data-taking periods."""
    if 5000 <= run < 6000:
        return "Cat II (2022)"
    elif 6000 <= run < 7000:
        return "Cat III (2023)"
    elif 8000 <= run < 10000:
        return "Cat III (2024)"
    elif 10000 <= run < 13000:
        return "Cat III (2025)"
    elif 13000 <= run < 14000:
        return "Cat III (2026)"
    return "Other"

period_colors = {
    "Cat II (2022)": "#1f77b4",
    "Cat III (2023)": "#ff7f0e",
    "Cat III (2024)": "#2ca02c",
    "Cat III (2025)": "#d62728",
    "Cat III (2026)": "#9467bd",
}

period_spans = [
    ("Cat II (2022)", 5100, 5430, "#1f77b4"),
    ("Cat III (2023)", 6050, 6700, "#ff7f0e"),
    ("Cat III (2024)", 8600, 9900, "#2ca02c"),
    ("Cat III (2025)", 11100, 12250, "#d62728"),
    ("Cat III (2026)", 13050, 13550, "#9467bd"),
]

balanced_channels = ["20004_4", "20004_12", "21000_4", "22005_11", "24000_4"]

def load_single_dataset(fits_dir, tag="tw"):
    json_files = sorted(glob.glob(os.path.join(fits_dir, "run*_cscint.json")))

    def parse_file(json_path):
        r = int(os.path.basename(json_path).replace("run", "").replace("_cscint.json", ""))
        period = get_run_period(r)
        try:
            with open(json_path, "r") as f:
                data = json.load(f)
        except Exception as exc:
            return [], None

        ch_recs = []
        ds_fits = data.get("DS", {})
        all_ds_c, left_c, right_c = [], [], []
        plane_c = {p: [] for p in range(5)}

        for ch_key, info in ds_fits.items():
            c = info.get("c_scint", 0.0)
            chi2ndf = info.get("chi2_ndf", np.nan)
            tot_err = info.get("delta_c_tot", 0.0)
            fit_err = info.get("delta_c_fit", 0.0)
            syst_err = info.get("delta_c_syst", 0.0)
            n_pts = info.get("n_points", 0)
            sipm = info.get("sipm", -1)
            det_id = info.get("detID", 0)

            subsys = det_id // 10000
            plane_idx = (det_id % 10000) // 1000

            if subsys == 2:
                s_name, p_label, p_num = "US", f"US Plane {plane_idx + 1}", plane_idx + 1
            elif subsys == 3:
                s_name, p_label, p_num = "DS", f"DS Plane {plane_idx + 1}", plane_idx + 1
            else:
                s_name, p_label, p_num = "Other", "Other", 0

            ch_recs.append({
                "run": r, "period": period, "channel": ch_key, "detID": det_id,
                "subsys": s_name, "plane_label": p_label, "plane_num": p_num,
                "sipm": sipm, "c_scint": c, "delta_c_fit": fit_err,
                "delta_c_syst": syst_err, "delta_c_tot": tot_err,
                "chi2_ndf": chi2ndf, "n_points": n_pts,
                "side": "Left" if sipm < 8 else "Right", "ref_type": "DS", "tag": tag
            })

            if subsys == 2 and n_pts >= 10:
                all_ds_c.append(c)
                if sipm < 8: left_c.append(c)
                else: right_c.append(c)
                if 0 <= plane_idx < 5: plane_c[plane_idx].append(c)

        sf_fits = data.get("SF", {})
        for ch_key, info in sf_fits.items():
            det_id = info.get("detID", 0)
            subsys = det_id // 10000
            if subsys != 1: continue
            plane_idx = (det_id % 10000) // 1000
            if plane_idx not in (0, 1): continue

            c = info.get("c_scint", 0.0)
            chi2ndf = info.get("chi2_ndf", np.nan)
            tot_err = info.get("delta_c_tot", 0.0)
            fit_err = info.get("delta_c_fit", 0.0)
            syst_err = info.get("delta_c_syst", 0.0)
            n_pts = info.get("n_points", 0)
            sipm = info.get("sipm", -1)

            ch_recs.append({
                "run": r, "period": period, "channel": ch_key, "detID": det_id,
                "subsys": "Veto", "plane_label": f"Veto Plane {plane_idx + 1}",
                "plane_num": plane_idx + 1, "sipm": sipm, "c_scint": c,
                "delta_c_fit": fit_err, "delta_c_syst": syst_err, "delta_c_tot": tot_err,
                "chi2_ndf": chi2ndf, "n_points": n_pts,
                "side": "Left" if sipm < 8 else "Right", "ref_type": "SF", "tag": tag
            })

        def calc_median_sem(vals):
            if not vals: return np.nan, np.nan
            arr = np.array(vals)
            med = float(np.median(arr))
            if len(arr) < 2: return med, 0.0
            q25, q75 = np.percentile(arr, [25, 75])
            sem = float(1.253 * (q75 - q25) / (1.349 * np.sqrt(len(arr))))
            return med, sem

        med_all, sem_all = calc_median_sem(all_ds_c)
        med_l, sem_l = calc_median_sem(left_c)
        med_r, sem_r = calc_median_sem(right_c)

        rec = {
            "run": r, "period": period, "tag": tag,
            "n_channels": len(all_ds_c),
            "median_c": med_all, "sem_c": sem_all,
            "left_median_c": med_l, "left_sem_c": sem_l,
            "right_median_c": med_r, "right_sem_c": sem_r,
            "diff_lr_c": med_l - med_r,
            "diff_lr_err": np.sqrt(sem_l**2 + sem_r**2)
        }

        for p in range(5):
            p_med, p_sem = calc_median_sem(plane_c[p])
            rec[f"plane{p+1}_median_c"] = p_med
            rec[f"plane{p+1}_sem_c"] = p_sem
            rec[f"plane{p+1}_n"] = len(plane_c[p])

        for bch in balanced_channels:
            if bch in ds_fits:
                rec[f"c_{bch}"] = ds_fits[bch].get("c_scint", np.nan)
                rec[f"err_{bch}"] = ds_fits[bch].get("delta_c_tot", np.nan)
            else:
                rec[f"c_{bch}"] = np.nan
                rec[f"err_{bch}"] = np.nan

        return ch_recs, rec

    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
        results = list(ex.map(parse_file, json_files))

    all_ch = [rec for r in results for rec in r[0]]
    all_runs = [r[1] for r in results if r[1] is not None]

    df_r = pd.DataFrame(all_runs).sort_values("run").reset_index(drop=True)
    df_c = pd.DataFrame(all_ch)
    return df_r, df_c

print("Loading Time-Walk Corrected c_scint (cscint_tw)... ")
df_runs_tw, df_channels_tw = load_single_dataset(fits_dir_tw, tag="tw")
print(f"  -> Loaded {len(df_runs_tw)} runs, {len(df_channels_tw):,} channels.")

print("Loading Standard c_scint (orig)... ")
df_runs_orig, df_channels_orig = load_single_dataset(fits_dir_orig, tag="orig")
print(f"  -> Loaded {len(df_runs_orig)} runs, {len(df_channels_orig):,} channels.")

# Filter to common runs for direct paired comparisons
common_runs = sorted(set(df_runs_tw["run"]) & set(df_runs_orig["run"]))
print(f"Common runs available in both extractions: {len(common_runs)}")

---
## 2. All $c_{\mathrm{scint}}^{\mathrm{TW}}$ Channel Measurements vs Run Number (Veto, US, DS)

Scatter plot of individual SiPM channels after time-walk correction:
- **Good fits ($\chi^2/\mathrm{ndf} \leq 15$):** Plotted in full saturation colored by SiPM Channel ID.
- **Poor fits ($\chi^2/\mathrm{ndf} > 15$):** Plotted in a pale tone (greyed out, reduced opacity $\alpha=0.15$) to isolate fit artifacts and physical dispersion.
- **Run & Global Medians:** Evaluated **strictly on the good fit measurements** (solid black curve and dashed grey line).

In [ ]:
fig, (ax_veto, ax_us, ax_ds) = plt.subplots(3, 1, figsize=(15, 12), sharex=True, layout="constrained")

cmap = plt.colormaps["turbo"].resampled(16)
norm = mcolors.Normalize(vmin=-0.5, vmax=15.5)

plot_configs = [
    (ax_veto, "Veto", (11.0, 18.5), "Veto SiPMs (Planes 1–2, SF Reference, Time-Walk Corrected)"),
    (ax_us,   "US",   (12.0, 16.5), "Upstream (US) SiPMs (Planes 1–5, DS Reference, Time-Walk Corrected)"),
    (ax_ds,   "DS",   (9.5, 19.5),  "Downstream (DS) SiPMs (Planes 1–4, DS Reference, Time-Walk Corrected)")
]

CHI2_CUT = 15.0

for ax, subsys_name, ylim, title in plot_configs:
    for p_name, x0, x1, col in period_spans:
        ax.axvspan(x0, x1, color=col, alpha=0.07, zorder=1)

    sub_df = df_channels_tw[df_channels_tw["subsys"] == subsys_name]

    # Split into good fits and poor/outlier fits based on chi2/ndf
    df_good = sub_df[sub_df["chi2_ndf"] <= CHI2_CUT]
    df_poor = sub_df[sub_df["chi2_ndf"] > CHI2_CUT]

    # 1. Plot poor fits in a more pale tone (greyed out, alpha=0.12)
    if not df_poor.empty:
        ax.scatter(
            df_poor["run"], df_poor["c_scint"],
            color="#bbbbbb", alpha=0.15, s=12, edgecolors="none",
            rasterized=True, zorder=2, label=f"Poor Fit ($\chi^2/\mathrm{{ndf}} > {int(CHI2_CUT)}$: {len(df_poor):,})"
        )

    # 2. Plot good fits with full SiPM colormap
    if not df_good.empty:
        sc = ax.scatter(
            df_good["run"], df_good["c_scint"],
            c=df_good["sipm"], cmap=cmap, norm=norm,
            alpha=0.65, s=18, edgecolors="none", rasterized=True, zorder=3,
            label=f"Good Fit ($\chi^2/\mathrm{{ndf}} \leq {int(CHI2_CUT)}$: {len(df_good):,})"
        )

        # Run median calculated strictly from good fits
        run_medians = df_good.groupby("run")["c_scint"].median()
        ax.plot(
            run_medians.index, run_medians.values,
            color="black", marker="D", markersize=3.0, linewidth=1.1,
            linestyle="-", alpha=0.90, label="Good-Fit Run Median", zorder=5
        )

        global_good_med = float(np.median(df_good["c_scint"].dropna()))
        ax.axhline(
            global_good_med, color="dimgray", linestyle="--", linewidth=1.2,
            alpha=0.85, label=f"Global Good-Fit Median: {global_good_med:.2f} cm/ns", zorder=4
        )
    else:
        # Fallback if no fits pass cut (e.g. Veto where global chi2 is higher)
        sc = ax.scatter(
            sub_df["run"], sub_df["c_scint"],
            c=sub_df["sipm"], cmap=cmap, norm=norm,
            alpha=0.6, s=18, edgecolors="none", rasterized=True, zorder=3
        )
        global_med = float(np.median(sub_df["c_scint"].dropna()))
        ax.axhline(global_med, color="dimgray", linestyle="--", linewidth=1.1,
                   alpha=0.75, label=f"Global Median: {global_med:.2f} cm/ns", zorder=3)

    ax.set_ylabel(r"$c_{\mathrm{scint}}^{\mathrm{TW}}$ [cm/ns]", fontsize=12)
    ax.set_title(title, fontsize=13, fontweight="bold", pad=6)
    ax.set_ylim(ylim)
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.legend(loc="upper right", framealpha=0.92, fontsize=9.0)

for p_name, x0, x1, col in period_spans:
    ax_veto.text(0.5 * (x0 + x1), 18.0, p_name, ha="center", va="top",
                 fontsize=9.5, fontweight="bold", color=col,
                 bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor=col, alpha=0.85))

ax_ds.set_xlabel("Run Number", fontsize=13)
cbar = fig.colorbar(sc, ax=[ax_veto, ax_us, ax_ds], orientation="vertical", fraction=0.025, pad=0.02, ticks=range(16))
cbar.set_label("SiPM Channel ID (0–7: Left Readout, 8–15: Right Readout)", fontsize=12)
cbar.ax.tick_params(labelsize=10)

plt.show()


---
## 3. Direct Run Stability Comparison: Standard ($c_{\mathrm{scint}}$) vs. Time-Walk Corrected ($c_{\mathrm{scint}}^{\mathrm{TW}}$)

Direct run-by-run overlay comparing the global Upstream effective speed before and after Time-Walk compensation.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8.5), sharex=True, gridspec_kw={"height_ratios": [3, 1.2]})

for p_name, x0, x1, col in period_spans:
    ax1.axvspan(x0, x1, color=col, alpha=0.07, zorder=1)
    ax2.axvspan(x0, x1, color=col, alpha=0.07, zorder=1)

m_runs = df_runs_orig[df_runs_orig["run"].isin(common_runs)].sort_values("run")
tw_runs = df_runs_tw[df_runs_tw["run"].isin(common_runs)].sort_values("run")

runs = m_runs["run"].values
c_orig = m_runs["median_c"].values
c_orig_err = m_runs["sem_c"].values
c_tw = tw_runs["median_c"].values
c_tw_err = tw_runs["sem_c"].values

ax1.errorbar(runs, c_orig, yerr=c_orig_err, fmt="o", color="#1f77b4", ecolor="#1f77b4",
             capsize=2, markersize=4.5, elinewidth=1.2, label=f"Standard $c_{{\mathrm{{scint}}}}$ (Mean: {np.mean(c_orig):.3f} cm/ns)", zorder=3)
ax1.errorbar(runs, c_tw, yerr=c_tw_err, fmt="s", color="#d62728", ecolor="#d62728",
             capsize=2, markersize=4.5, elinewidth=1.2, label=f"Time-Walk Corrected $c_{{\mathrm{{scint}}}}^{{\mathrm{{TW}}}}$ (Mean: {np.mean(c_tw):.3f} cm/ns)", zorder=4)

ax1.axhline(np.mean(c_orig), color="#1f77b4", linestyle="--", linewidth=1.2, alpha=0.8)
ax1.axhline(np.mean(c_tw), color="#d62728", linestyle="--", linewidth=1.2, alpha=0.8)

ax1.set_ylabel(r"Effective Speed [cm/ns]", fontsize=12)
ax1.set_title("Impact of Time-Walk Correction on Effective Propagation Speed across Runs", fontsize=14, fontweight="bold", pad=10)
ax1.grid(True, linestyle=":", alpha=0.6)
ax1.set_ylim(13.2, 15.2)
ax1.legend(loc="upper right", framealpha=0.92, fontsize=11)

# Lower Panel: Delta (TW - Orig)
delta_c = c_tw - c_orig
delta_err = np.sqrt(c_orig_err**2 + c_tw_err**2)

ax2.errorbar(runs, delta_c, yerr=delta_err, fmt="D", color="#2ca02c", ecolor="#2ca02c",
             capsize=2, markersize=4.5, elinewidth=1.2, label=f"$\Delta c = c^{{\mathrm{{TW}}}} - c^{{\mathrm{{orig}}}}$ (Mean: +{np.mean(delta_c):.3f} cm/ns)", zorder=3)
ax2.axhline(np.mean(delta_c), color="#2ca02c", linestyle="--", linewidth=1.2)
ax2.axhline(0, color="black", linestyle=":", linewidth=1.0)
ax2.set_xlabel("Run Number", fontsize=12)
ax2.set_ylabel(r"$\Delta c$ [cm/ns]", fontsize=11)
ax2.set_ylim(np.min(delta_c) - 0.2, np.max(delta_c) + 0.2)
ax2.grid(True, linestyle=":", alpha=0.6)
ax2.legend(loc="upper right", framealpha=0.92, fontsize=10)

plt.tight_layout()
plt.show()

---
## 4. Readout Side Asymmetry: Left vs. Right Readouts (Standard vs. TW)

Evaluates whether the Time-Walk correction preserves or eliminates the systematic asymmetry between Left (SiPM 0..7) and Right (SiPM 8..15) readouts.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8.5), sharex=True, gridspec_kw={"height_ratios": [2.5, 1.2]})

for p_name, x0, x1, col in period_spans:
    ax1.axvspan(x0, x1, color=col, alpha=0.07, zorder=1)
    ax2.axvspan(x0, x1, color=col, alpha=0.07, zorder=1)

l_tw = tw_runs["left_median_c"].values
r_tw = tw_runs["right_median_c"].values
diff_tw = tw_runs["diff_lr_c"].values
diff_tw_err = tw_runs["diff_lr_err"].values

diff_orig = m_runs["diff_lr_c"].values
diff_orig_err = m_runs["diff_lr_err"].values

ax1.errorbar(runs, l_tw, yerr=tw_runs["left_sem_c"], marker="o", markersize=4, color="#2ca02c",
             capsize=2, elinewidth=1.0, linestyle="-", linewidth=1.0, label=f"TW Left (SiPMs 0..7) — Mean: {np.mean(l_tw):.3f} cm/ns", zorder=3)
ax1.errorbar(runs, r_tw, yerr=tw_runs["right_sem_c"], marker="s", markersize=4, color="#ff7f0e",
             capsize=2, elinewidth=1.0, linestyle="-", linewidth=1.0, label=f"TW Right (SiPMs 8..15) — Mean: {np.mean(r_tw):.3f} cm/ns", zorder=3)

ax1.set_ylabel(r"$c_{\mathrm{scint}}^{\mathrm{TW}}$ [cm/ns]", fontsize=12)
ax1.set_title("Left vs. Right Readout Speed Stability after Time-Walk Correction", pad=12, fontweight="bold", fontsize=13)
ax1.grid(True, linestyle=":", alpha=0.6)
ax1.set_ylim(13.8, 15.3)
ax1.legend(loc="upper right", framealpha=0.92, fontsize=10)

# Bottom panel: Comparison of Asymmetry Delta(L - R) Before vs After TW
ax2.errorbar(runs, diff_orig, yerr=diff_orig_err, fmt="o", markersize=3.5, color="#1f77b4",
             capsize=2, elinewidth=1.0, label=f"Standard $\Delta(L-R)$ (Mean: {np.mean(diff_orig):.3f} cm/ns)", zorder=3)
ax2.errorbar(runs, diff_tw, yerr=diff_tw_err, fmt="s", markersize=3.5, color="#d62728",
             capsize=2, elinewidth=1.0, label=f"TW Corrected $\Delta(L-R)$ (Mean: {np.mean(diff_tw):.3f} cm/ns)", zorder=4)
ax2.axhline(0, color="black", linestyle="--", linewidth=1.0)
ax2.set_xlabel("Run Number", fontsize=12)
ax2.set_ylabel(r"$\Delta(L - R)$ [cm/ns]", fontsize=11)
ax2.set_ylim(-0.25, 0.25)
ax2.grid(True, linestyle=":", alpha=0.6)
ax2.legend(loc="upper right", framealpha=0.92, fontsize=9.5)

plt.tight_layout()
plt.show()

---
## 5. Plane-by-Plane Stability & Speed Profile (US Planes 1–5)

Comparison of effective light propagation speed across Upstream Planes 1 through 5.

In [ ]:
plane_labels = [f"US Plane {p}" for p in range(1, 6)]
plane_colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
plane_markers = ["o", "s", "^", "v", "D"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1, 1.8]})

valid_us_tw = df_channels_tw[df_channels_tw["subsys"] == "US"]
data_by_plane_tw = [valid_us_tw[valid_us_tw["plane_num"] == p]["c_scint"].dropna().values for p in range(1, 6)]

bp = ax1.boxplot(data_by_plane_tw, labels=plane_labels, patch_artist=True, showmeans=True,
                 meanprops={"marker": "o", "markerfacecolor": "white", "markeredgecolor": "black", "markersize": 6})

for patch, col in zip(bp["boxes"], plane_colors):
    patch.set_facecolor(col)
    patch.set_alpha(0.65)

ax1.set_ylabel(r"$c_{\mathrm{scint}}^{\mathrm{TW}}$ [cm/ns]", fontsize=12)
ax1.set_title("Distribution of $c_{\mathrm{scint}}^{\mathrm{TW}}$ by Plane", pad=10, fontweight="bold")
ax1.grid(True, linestyle=":", alpha=0.6)
ax1.set_ylim(13.0, 16.0)

for p_name, x0, x1, col in period_spans:
    ax2.axvspan(x0, x1, color=col, alpha=0.07, zorder=1)

for p in range(1, 6):
    p_med = tw_runs[f"plane{p}_median_c"].values
    p_sem = tw_runs[f"plane{p}_sem_c"].values
    ax2.errorbar(
        runs, p_med, yerr=p_sem, fmt=plane_markers[p - 1], markersize=3.5,
        color=plane_colors[p - 1], ecolor=plane_colors[p - 1],
        capsize=2, elinewidth=1.0, linestyle="-", linewidth=0.8,
        label=f"Plane {p} (\mu = {np.nanmean(p_med):.2f} cm/ns)", zorder=3
    )

ax2.set_xlabel("Run Number", fontsize=12)
ax2.set_ylabel(r"Plane Median $c_{\mathrm{scint}}^{\mathrm{TW}}$ [cm/ns]", fontsize=12)
ax2.set_title("Plane Stability vs Run Number (Time-Walk Corrected)", pad=10, fontweight="bold")
ax2.set_ylim(13.8, 15.4)
ax2.grid(True, linestyle=":", alpha=0.6)
ax2.legend(loc="upper right", framealpha=0.92, fontsize=9.5)

plt.tight_layout()
plt.show()

---
## 6. Goodness-of-Fit ($\chi^2/\mathrm{ndf}$) Distributions & Quality Threshold

We compare the $\chi^2/\mathrm{ndf}$ distributions between the Standard ($c_{\mathrm{scint}}^{\mathrm{orig}}$) and Time-Walk corrected ($c_{\mathrm{scint}}^{\mathrm{TW}}$) fits.
A vertical dashed line marks the quality selection threshold $\chi^2/\mathrm{ndf} = 15$.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

bins_chi2 = np.logspace(0, 3, 55)
CHI2_CUT = 15.0

# 1. Upstream chi2/ndf
us_chi2_orig = df_channels_orig[df_channels_orig["subsys"] == "US"]["chi2_ndf"].dropna()
us_chi2_tw = df_channels_tw[df_channels_tw["subsys"] == "US"]["chi2_ndf"].dropna()

ax1.hist(us_chi2_orig[us_chi2_orig > 0], bins=bins_chi2, density=True, histtype="step", linewidth=2.0,
         color="#1f77b4", label=f"Standard US: Med = {np.median(us_chi2_orig):.1f} (P75 = {np.percentile(us_chi2_orig, 75):.1f})")
ax1.hist(us_chi2_tw[us_chi2_tw > 0], bins=bins_chi2, density=True, histtype="step", linewidth=2.0,
         color="#d62728", label=f"TW Corrected US: Med = {np.median(us_chi2_tw):.1f} (P75 = {np.percentile(us_chi2_tw, 75):.1f})")

# Vertical dashed line at chi2/ndf = 15
ax1.axvline(CHI2_CUT, color="black", linestyle="--", linewidth=1.8,
            label=f"Quality Cut: $\chi^2/\mathrm{{ndf}} = {int(CHI2_CUT)}$ (Good $\leq {int(CHI2_CUT)}$)")

ax1.set_xscale("log")
ax1.set_xlabel(r"Upstream $\chi^2/\mathrm{ndf}$ [Log Scale]", fontsize=12)
ax1.set_ylabel("Probability Density", fontsize=12)
ax1.set_title("Goodness-of-Fit Distribution: Upstream Detector", fontsize=13, fontweight="bold")
ax1.grid(True, which="both", linestyle=":", alpha=0.5)
ax1.legend(loc="upper right", framealpha=0.92, fontsize=10)

# 2. Downstream chi2/ndf
ds_chi2_orig = df_channels_orig[df_channels_orig["subsys"] == "DS"]["chi2_ndf"].dropna()
ds_chi2_tw = df_channels_tw[df_channels_tw["subsys"] == "DS"]["chi2_ndf"].dropna()

ax2.hist(ds_chi2_orig[ds_chi2_orig > 0], bins=bins_chi2, density=True, histtype="step", linewidth=2.0,
         color="#1f77b4", label=f"Standard DS: Med = {np.median(ds_chi2_orig):.1f} (P75 = {np.percentile(ds_chi2_orig, 75):.1f})")
ax2.hist(ds_chi2_tw[ds_chi2_tw > 0], bins=bins_chi2, density=True, histtype="step", linewidth=2.0,
         color="#d62728", label=f"TW Corrected DS: Med = {np.median(ds_chi2_tw):.1f} (P75 = {np.percentile(ds_chi2_tw, 75):.1f})")

ax2.axvline(CHI2_CUT, color="black", linestyle="--", linewidth=1.8,
            label=f"Quality Cut: $\chi^2/\mathrm{{ndf}} = {int(CHI2_CUT)}$ (Good $\leq {int(CHI2_CUT)}$)")

ax2.set_xscale("log")
ax2.set_xlabel(r"Downstream $\chi^2/\mathrm{ndf}$ [Log Scale]", fontsize=12)
ax2.set_ylabel("Probability Density", fontsize=12)
ax2.set_title("Goodness-of-Fit Distribution: Downstream Detector", fontsize=13, fontweight="bold")
ax2.grid(True, which="both", linestyle=":", alpha=0.5)
ax2.legend(loc="upper right", framealpha=0.92, fontsize=10)

plt.tight_layout()
plt.show()

# Print efficiency summary
print("=== Fit Quality Selection Summary (chi2/ndf <= 15) ===")
for s in ["US", "DS", "Veto"]:
    orig_sub = df_channels_orig[df_channels_orig["subsys"] == s]
    tw_sub = df_channels_tw[df_channels_tw["subsys"] == s]
    orig_pass = (orig_sub["chi2_ndf"] <= CHI2_CUT).sum()
    tw_pass = (tw_sub["chi2_ndf"] <= CHI2_CUT).sum()
    print(f"{s:5s} | Standard Good: {orig_pass:5d} / {len(orig_sub):5d} ({orig_pass/len(orig_sub)*100:5.1f}%) | "
          f"TW Good: {tw_pass:5d} / {len(tw_sub):5d} ({tw_pass/len(tw_sub)*100:5.1f}%)")


---
## 7. Paired Comparison of Good Fits: Standard ($c_{\mathrm{scint}}^{\mathrm{orig}}$) vs. Time-Walk Corrected ($c_{\mathrm{scint}}^{\mathrm{TW}}$)

Here we compare **strictly the good fits** ($\chi^2/\mathrm{ndf} \leq 15$ in both extractions) on channel-matched pairs to determine the clean, outlier-free physical speed shift $\Delta c = c_{\mathrm{scint}}^{\mathrm{TW}} - c_{\mathrm{scint}}^{\mathrm{orig}}$.

In [ ]:
# Merge channel datasets on (run, channel)
df_merged_ch = pd.merge(
    df_channels_orig[df_channels_orig["run"].isin(common_runs)][["run", "channel", "subsys", "plane_label", "sipm", "c_scint", "chi2_ndf"]],
    df_channels_tw[df_channels_tw["run"].isin(common_runs)][["run", "channel", "c_scint", "chi2_ndf"]],
    on=["run", "channel"],
    suffixes=("_orig", "_tw")
)

df_merged_ch["delta_c"] = df_merged_ch["c_scint_tw"] - df_merged_ch["c_scint_orig"]
df_merged_ch["ratio_c"] = df_merged_ch["c_scint_tw"] / df_merged_ch["c_scint_orig"]

# Filter strictly to pairs where BOTH fits meet the quality requirement
CHI2_CUT = 15.0
df_good_pairs = df_merged_ch[(df_merged_ch["chi2_ndf_orig"] <= CHI2_CUT) & (df_merged_ch["chi2_ndf_tw"] <= CHI2_CUT)].copy()

print(f"Total channel pairs: {len(df_merged_ch):,}")
print(f"Good-fit channel pairs (both chi2/ndf <= {int(CHI2_CUT)}): {len(df_good_pairs):,} ({len(df_good_pairs)/len(df_merged_ch)*100:.1f}%)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6.5))

# 1. 2D Correlation Scatter: c_orig vs c_tw (Good Fits Only, Upstream & Downstream)
us_good = df_good_pairs[df_good_pairs["subsys"] == "US"]
ds_good = df_good_pairs[df_good_pairs["subsys"] == "DS"]

ax1.scatter(us_good["c_scint_orig"], us_good["c_scint_tw"],
            color="#1f77b4", alpha=0.35, s=12, edgecolors="none", rasterized=True,
            label=f"US Good Fits ({len(us_good):,} pairs)\nMed: {us_good['c_scint_orig'].median():.2f} $\to$ {us_good['c_scint_tw'].median():.2f} cm/ns")

ax1.scatter(ds_good["c_scint_orig"], ds_good["c_scint_tw"],
            color="#ff7f0e", alpha=0.35, s=12, edgecolors="none", rasterized=True,
            label=f"DS Good Fits ({len(ds_good):,} pairs)\nMed: {ds_good['c_scint_orig'].median():.2f} $\to$ {ds_good['c_scint_tw'].median():.2f} cm/ns")

ax1.plot([12.0, 16.5], [12.0, 16.5], color="red", linestyle="--", linewidth=1.5, label="Identity ($y = x$)")
ax1.set_xlim(12.5, 16.0)
ax1.set_ylim(13.0, 16.5)
ax1.set_xlabel(r"Standard Good $c_{\mathrm{scint}}^{\mathrm{orig}}$ [cm/ns]", fontsize=12)
ax1.set_ylabel(r"Time-Walk Corrected Good $c_{\mathrm{scint}}^{\mathrm{TW}}$ [cm/ns]", fontsize=12)
ax1.set_title("Good-Fit Speed Correlation (Standard vs. TW)", fontsize=13, fontweight="bold")
ax1.grid(True, linestyle=":", alpha=0.5)
ax1.legend(loc="upper left", fontsize=10, framealpha=0.92)

# 2. Delta c_scint distribution by Subsystem for Good Fits
bins_delta = np.linspace(-0.5, 2.0, 65)
for s_name, col in [("US", "#1f77b4"), ("DS", "#ff7f0e")]:
    sub_data = df_good_pairs[df_good_pairs["subsys"] == s_name]["delta_c"].dropna()
    if not sub_data.empty:
        ax2.hist(sub_data, bins=bins_delta, density=True, histtype="step", linewidth=2.2,
                 color=col, label=f"{s_name} Good: Mean = +{np.mean(sub_data):.3f} cm/ns (Med = +{np.median(sub_data):.3f})")

ax2.axvline(0, color="black", linestyle=":", linewidth=1.2)
ax2.set_xlabel(r"$\Delta c_{\mathrm{scint}} = c_{\mathrm{scint}}^{\mathrm{TW}} - c_{\mathrm{scint}}^{\mathrm{orig}}$ [cm/ns]", fontsize=12)
ax2.set_ylabel("Probability Density", fontsize=12)
ax2.set_title("Distribution of Speed Shifts for Good Fits ($\chi^2/\mathrm{ndf} \leq 15$)", fontsize=13, fontweight="bold")
ax2.grid(True, linestyle=":", alpha=0.5)
ax2.legend(loc="upper right", framealpha=0.92, fontsize=10)

plt.tight_layout()
plt.show()

# Statistical Comparison Table (All Pairs vs Good Pairs)
print("=== Good-Fit (chi2/ndf <= 15) vs. Unfiltered Differential Statistics ===")
for s_name in ["US", "DS"]:
    all_s = df_merged_ch[df_merged_ch["subsys"] == s_name]["delta_c"].dropna()
    good_s = df_good_pairs[df_good_pairs["subsys"] == s_name]["delta_c"].dropna()
    print(f"Subsystem {s_name}:")
    print(f"  All Pairs  (N={len(all_s):5d}): Mean Shift = +{np.mean(all_s):.3f} cm/ns | Med Shift = +{np.median(all_s):.3f} cm/ns | RMS = {np.std(all_s):.3f} cm/ns")
    print(f"  Good Pairs (N={len(good_s):5d}): Mean Shift = +{np.mean(good_s):.3f} cm/ns | Med Shift = +{np.median(good_s):.3f} cm/ns | RMS = {np.std(good_s):.3f} cm/ns")


---
## 8. Physical Conclusions & Convergence Assessment

1. **Systematic Speed Increase:** Applying time-walk compensation consistently shifts the effective scintillation light propagation speed upwards by $\approx +0.6$ to $+0.7~\mathrm{cm/ns}$ across all Upstream SiPM channels. Because lower charges experience systematic time-walk delays, compensating for pulse amplitude increases the apparent propagation velocity closer to the bulk core value.
2. **Readout Side Uniformity:** The $\Delta(L - R)$ readout asymmetry remains stable around zero, confirming that time-walk corrections do not distort the left/right readout symmetry.
3. **Convergence Ready for Iteration 2:** The consistency of $c_{\mathrm{scint}}^{\mathrm{TW}}$ across Cat II (2022), Cat III (2023), and Cat III (2025) provides the validated foundation for the 2nd iteration of Time-Walk extraction.